<a href="https://colab.research.google.com/github/grsart/BiomolComp/blob/main/Pratica6/AnaliseFilogenetica_PlanoB_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análise Filogenética – Partes 1 e 2 sem o MEGA (Plano B) #

Use este notebook **somente se o MEGA não estiver disponível** no computador. Ele reproduz as Partes 1 e 2 da prática no Google Colab, com as **mesmas questões (Q1 a Q15)**, de forma que a folha de respostas é a mesma.

Diferenças em relação ao MEGA:

* o alinhamento também é feito com o ClustalW (mesmas penalidades de gap);
* a árvore Neighbor-Joining usa distâncias calculadas com a matriz **BLOSUM62** (no MEGA usamos o modelo JTT), por isso os comprimentos de ramo e alguns valores de bootstrap podem variar um pouco;
* em vez de clicar na árvore para enraizá-la, o enraizamento é feito por código.

Para salvar uma árvore (os "prints" pedidos na apostila), clique com o botão direito na imagem e escolha **Salvar imagem como...**.

Execute as células na ordem, de cima para baixo.

In [ ]:
#@title Preparando o ambiente para a execução (leva cerca de 1 minuto)

!pip3 install -q biopython==1.85
!apt-get install -y -qq clustalw > /dev/null

import shutil
import subprocess
import urllib.request

import matplotlib.pyplot as plt
from IPython.display import HTML, display
from Bio import SeqIO, AlignIO, Phylo
from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor
from Bio.Phylo.Consensus import bootstrap_trees, get_support

assert shutil.which("clustalw"), "ClustalW não foi instalado"


def alinhar(arquivo_fasta):
    """Alinha as sequências com o ClustalW (abertura de gap 5.0, extensão 0.1)."""
    result = subprocess.run(
        ["clustalw", f"-INFILE={arquivo_fasta}", "-GAPOPEN=5", "-GAPEXT=0.1"],
        capture_output=True, text=True)
    assert result.returncode == 0, result.stdout + result.stderr
    return AlignIO.read(arquivo_fasta.rsplit(".", 1)[0] + ".aln", "clustal")


# Cores por propriedade físico-química, semelhantes às do esquema ClustalX
CORES = {**dict.fromkeys("AILMFWV", "#80a0f0"), **dict.fromkeys("KR", "#f01505"),
         **dict.fromkeys("DE", "#c048c0"), **dict.fromkeys("NQST", "#15c015"),
         "C": "#f08080", "G": "#f09048", "P": "#c0c000", **dict.fromkeys("HY", "#15a4a4")}


def mostrar_alinhamento(alinhamento, inicio=0, fim=None):
    """Mostra o trecho [inicio, fim) do alinhamento, colorindo os resíduos."""
    fim = fim or alinhamento.get_alignment_length()
    linhas = []
    for record in alinhamento:
        residuos = "".join(
            f'<span style="background:{CORES.get(aa, "white")}">{aa}</span>'
            for aa in str(record.seq[inicio:fim]))
        linhas.append(f"{record.id:<16}{residuos}")
    # Régua com o número da coluna (contando a partir de 1) a cada 10 posições
    regua = [" "] * (fim - inicio)
    for i in range(inicio, fim):
        if (i + 1) % 10 == 0:
            rotulo = str(i + 1)
            regua[i - inicio:i - inicio + len(rotulo)] = rotulo
    regua = " " * 16 + "".join(regua)[:fim - inicio]
    display(HTML('<pre style="font-size:12px; line-height:1.2">' + regua + "\n" + "\n".join(linhas) + "</pre>"))


def arvore_nj(alinhamento, n_replicas=100):
    """Árvore Neighbor-Joining (BLOSUM62) com suporte de bootstrap em porcentagem."""
    constructor = DistanceTreeConstructor(DistanceCalculator("blosum62"), "nj")
    tree = constructor.build_tree(alinhamento)
    replicas = list(bootstrap_trees(alinhamento, n_replicas, constructor))
    tree = get_support(tree, replicas)
    for clade in tree.get_nonterminals():
        clade.name = None
        if clade.confidence is not None:
            clade.confidence = round(clade.confidence)
    return tree


def enraizar(tree, outgroup):
    """Enraíza a árvore no grupo externo indicado."""
    tree.root_with_outgroup(outgroup)
    # O ramo da raiz separa o grupo externo de todo o resto: o suporte dele não é informativo
    for clade in tree.root.clades:
        clade.confidence = None
    tree.ladderize()
    return tree


def desenhar(tree, titulo, destacar=()):
    """Desenha a árvore com os valores de bootstrap; os nomes em `destacar` ficam em vermelho."""
    fig, ax = plt.subplots(figsize=(10, 11))
    Phylo.draw(tree, axes=ax, do_show=False,
               label_colors={nome: "red" for nome in destacar},
               branch_labels=lambda c: c.confidence if c.confidence is not None and not c.is_terminal() else None)
    ax.set_title(titulo)
    plt.tight_layout()
    plt.show()


print("Ambiente pronto.")

# Parte 1 #

## 2- Análise de alinhamento ##

Vamos baixar o arquivo **filogenia1.fasta**, com as sequências do domínio de transcriptase reversa (RT) de retrotransposons non-LTR.

In [ ]:
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/grsart/BiomolComp/main/Pratica6/filogenia1.fasta",
    "filogenia1.fasta")

records = list(SeqIO.parse("filogenia1.fasta", "fasta"))
print(f"{'SeqID':<12} {'Tamanho':>8}")
for rec in records:
    print(f"{rec.id:<12} {len(rec):>8}")
print(f"\nNúmero de sequências: {len(records)}")

# Primeiros 80 resíduos de cada sequência, ainda sem alinhar
mostrar_alinhamento(records, 0, 80)

Acima estão os tamanhos das sequências e os 80 primeiros resíduos de cada uma, **antes** do alinhamento.

**Q1- São quantas sequências? As sequências possuem o mesmo tamanho? Elas estão alinhadas? Se sim, como está o alinhamento?**

Agora vamos alinhar as sequências com o ClustalW, usando **Gap Opening Penalty = 5.0** e **Gap Extension Penalty = 0.1**.

*Por que esses valores?* Os valores padrão do ClustalW (10 e 0.2) penalizam muito a abertura de gaps. As famílias de retrotransposons divergiram há muito tempo e o domínio RT acumula inserções e deleções entre famílias. Penalidades menores permitem que o programa abra os gaps necessários para manter alinhados os motivos conservados.

In [ ]:
align1 = alinhar("filogenia1.fasta")
print(f"Alinhamento com {len(align1)} sequências e {align1.get_alignment_length()} colunas")

# Mostra o alinhamento completo, em blocos de 100 colunas
for inicio in range(0, align1.get_alignment_length(), 100):
    mostrar_alinhamento(align1, inicio, inicio + 100)

**Q2- Os motivos conservados da imagem abaixo estão presentes no alinhamento obtido?**

*Dica:* procure no alinhamento a sequência **FADD** nas sequências L1 (variação do motivo catalítico YADD da transcriptase reversa) e o motivo **QG...SP** que aparece algumas posições antes. A partir deles, localize os demais motivos da figura. A régua acima do alinhamento mostra o número das colunas.

<img width="800" alt="Motivos conservados do domínio de transcriptase reversa" src="https://github.com/user-attachments/assets/7b45e116-51c7-4695-9509-816c4ecd9929" />

## 3- Construção da árvore filogenética ##

Vamos construir a árvore pelo método **Neighbor-Joining**, com **teste de bootstrap de 100 réplicas**.

**Q3- Qual a importância de fazer o teste de bootstrap, em vez de construir apenas a árvore (sem teste de filogenia)? O que significa o número que aparece em cada ramo?**

**Atenção:** a célula abaixo leva de 1 a 3 minutos.

In [ ]:
arvore1 = arvore_nj(align1, n_replicas=100)
desenhar(arvore1, "Neighbor-Joining (BLOSUM62), bootstrap 100 réplicas – sem raiz definida")

## 4- Análise da árvore filogenética ##

**Q4- A árvore está enraizada? Se sim, qual é a raiz?**
*Lembre que o método Neighbor-Joining produz uma árvore sem raiz. O programa apenas desenha a árvore a partir de um ponto qualquer.*

Salve esta árvore (botão direito > Salvar imagem como...).

**Q5- Antes de enraizar: olhando a árvore, onde você acha que a raiz deveria estar? Que informação você precisaria para decidir?**

Vamos definir um enraizamento a partir de uma informação conhecida. Sabe-se que o transposon CRE2 representa a primeira família a se divergir das outras.

**Q6- Essa informação é suficiente para definir essa família como a raiz da árvore? Explique utilizando o conceito de grupo externo (*outgroup*).**

In [ ]:
arvore1 = enraizar(arvore1, "CRE2")
desenhar(arvore1, "Neighbor-Joining (BLOSUM62), bootstrap 100 réplicas – raiz no CRE2")

Salve a árvore enraizada.

**Q7- O que mudou ao fazer o enraizamento? A topologia (quem agrupa com quem) mudou? E a interpretação da árvore?**

Compare a árvore obtida com a árvore de referência abaixo. Cada clado na figura representa uma família distinta (por exemplo L1, RTE, R2, R4, CR1, Jockey, I, R1, Tad1 e CRE).

**Q8- Os membros na árvore da prática estão agrupados da mesma forma que na figura abaixo? Indique as diferenças e verifique o valor de bootstrap dos ramos que diferem.**

**Q9- A partir da árvore obtida, os elementos z82275, pido e Sr1 fazem parte de qual família? Justifique usando a topologia e os valores de bootstrap.**

<img width="900" alt="Árvore de referência das famílias de retrotransposons non-LTR" src="https://github.com/user-attachments/assets/725c6933-9a62-4def-9b12-8ca432e5e5a7" />

# Parte 2 #

## Identificação de família para duas proteínas inéditas ##

Faça os **passos 1 a 11 da apostila da Parte 2** no site do NCBI (eles não usam o MEGA) e responda às questões Q10, Q11 e Q12.

Ao final, você terá as duas sequências do domínio de transcriptase reversa, de **CAJ00246.1** e de **CAJ00242.1**, em formato FASTA.

Cole **somente as letras dos aminoácidos** de cada sequência (sem a linha de cabeçalho com `>`) entre as aspas triplas da célula abaixo e execute a célula. Quebras de linha e espaços são removidos automaticamente.

In [ ]:
TPA_poliprotein = """
COLE AQUI A SEQUÊNCIA DO DOMÍNIO RT DE CAJ00246.1
"""

TPA_Sm = """
COLE AQUI A SEQUÊNCIA DO DOMÍNIO RT DE CAJ00242.1
"""


def limpar(seq):
    return "".join(seq.split()).upper()


novas = {"TPA_poliprotein": limpar(TPA_poliprotein), "TPA_Sm": limpar(TPA_Sm)}
for nome, seq in novas.items():
    assert seq.isalpha() and "COLE" not in seq, f"Cole a sequência de {nome} na célula acima"
    print(f"{nome}: {len(seq)} aminoácidos")

# Monta o arquivo filogenia2.fasta: as duas novas sequências + as de filogenia1.fasta
with open("filogenia2.fasta", "w") as f:
    for nome, seq in novas.items():
        f.write(f">{nome}\n{seq}\n")
    SeqIO.write(records, f, "fasta")

print(f"\nfilogenia2.fasta criado com {len(list(SeqIO.parse('filogenia2.fasta', 'fasta')))} sequências")

Agora repetimos o procedimento da Parte 1 (alinhamento, árvore NJ com bootstrap e enraizamento no CRE2) para o arquivo **filogenia2.fasta**. As duas novas sequências aparecem **em vermelho**.

**Atenção:** a célula abaixo leva de 1 a 3 minutos.

In [ ]:
align2 = alinhar("filogenia2.fasta")
arvore2 = enraizar(arvore_nj(align2, n_replicas=100), "CRE2")
desenhar(arvore2, "filogenia2: Neighbor-Joining (BLOSUM62), bootstrap 100 réplicas – raiz no CRE2",
         destacar=novas.keys())

Salve a árvore.

**Q13- Essas proteínas foram classificadas em quais famílias?**

**Q14- Verifique o valor de bootstrap dos ramos que agrupam cada uma das duas sequências à sua família. O agrupamento é confiável? Explique.**
*Referência usual:* valores de bootstrap ≥ 70% costumam ser considerados bom suporte para um ramo; valores abaixo de 50% indicam que o agrupamento não é sustentado pelos dados.

**Q15- Essas duas sequências vêm de um mesmo organismo. Por que ficaram tão distantes uma da outra na árvore? Qual a explicação biológica para isso?**

---
Depois de terminar, siga para a **Parte 3** (notebook AnaliseFilogenetica3.ipynb).